# LAX — Fast CPU-only Google Colab launcher

This notebook launches LAX in CPU fast mode when no GPU is available. It uses the smaller `Qwen/Qwen2.5-0.5B-Instruct` model for generation, keeps the curated wellbeing retrieval and safety routing, and limits reply length to reduce waiting time.

**Important trade-off:** CPU fast mode does not load the saved 1.5B LoRA adapter, because that adapter cannot be applied to a different-sized model. This makes CPU inference faster, but answers may be less consistent than the fine-tuned model. To use the full fine-tuned model instead, set `LAX_CPU_FAST_MODE=0` and provide the saved adapter; it will be slower.


## 1. Configure fast CPU mode

The fast CPU path does not need Google Drive or the saved LoRA adapter. This avoids an unnecessary mount/upload step.

In [ ]:
from pathlib import Path

# Only needed if you later turn fast mode off; the default fast CPU mode skips it.
ADAPTER_PATH = Path("/content/drive/MyDrive/psychology-qwen-qlora-v2")
print("CPU fast mode enabled: using Qwen2.5-0.5B-Instruct without the LoRA adapter.")
print("The grounded retrieval, topic routing, and high-risk safety check remain enabled.")

## 2. Clone or update the GitHub repository

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/psychology-raft-rag-chatbot")
REPO_URL = "https://github.com/karthikeyan-crypto/psychology-raft-rag-chatbot.git"

if (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

assert (REPO_DIR / "app.py").is_file(), "app.py was not found in the repository."
assert (REPO_DIR / "requirements-colab-cpu.txt").is_file(), "CPU requirements file is missing."
print("Repository is ready:", REPO_DIR)

## 3. Force CPU fast mode before starting the app

This hides any available GPU from this CPU-only launcher and selects the smaller faster model. The saved LoRA adapter is not loaded in CPU fast mode.

In [ ]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["LAX_ADAPTER_PATH"] = str(ADAPTER_PATH)
os.environ["LAX_CPU_FAST_MODE"] = "1"
os.environ["LAX_LOAD_GENERAL_ON_CPU"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

%cd /content/psychology-raft-rag-chatbot
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CPU fast mode:", os.environ["LAX_CPU_FAST_MODE"])
assert not torch.cuda.is_available(), "CUDA is visible. Restart with a CPU runtime and run again."

## 4. Install the CPU-only app dependencies

This requirements file preserves Colab's existing PyTorch installation.

In [ ]:
%pip install -q -r /content/psychology-raft-rag-chatbot/requirements-colab-cpu.txt

## 5. Remove incompatible optional TorchAO

Some Colab runtimes include an older optional TorchAO package that Transformers rejects during model loading. LAX's CPU mode does not use TorchAO, so remove it only when the installed version is `0.16.0` or older.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

try:
    torchao_version = metadata.version("torchao")
except metadata.PackageNotFoundError:
    torchao_version = None

print("Existing TorchAO:", torchao_version)
if torchao_version is not None:
    # This notebook is CPU-only. LAX's CPU path does not use TorchAO, and
    # a mismatched optional binary can emit warnings or break model startup.
    print("Removing optional TorchAO from this CPU-only runtime.")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        check=True,
    )
else:
    print("TorchAO is absent; this is expected for CPU-only LAX.")

try:
    print("TorchAO after check:", metadata.version("torchao"))
except metadata.PackageNotFoundError:
    print("TorchAO is not installed (expected for this CPU-only app).")


## 6. Verify installed dependencies

In [ ]:
import importlib.metadata as metadata
import torch

for package in [
    "transformers", "peft", "accelerate",
    "sentence-transformers", "faiss-cpu", "streamlit", "safetensors"
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"MISSING: {package}")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

## 7. Build the local knowledge index if it does not exist

This uses the repository's included Markdown knowledge files.

In [ ]:
import subprocess
index_dir = REPO_DIR / "data" / "vector_index"
index_file = index_dir / "student_wellbeing.faiss"
metadata_file = index_dir / "metadata.json"

if not (index_file.is_file() and metadata_file.is_file()):
    subprocess.run(
        [
            sys.executable, "inference/rag_local.py",
            "--source-dir", "data/sources",
            "--index-dir", "data/vector_index",
        ],
        cwd=str(REPO_DIR),
        check=True,
    )
else:
    print("Existing local FAISS index found; keeping it.")

print("Index ready:", index_file.is_file() and metadata_file.is_file())

## 8. Start Streamlit and inspect its log

In [ ]:
import subprocess
import time
import urllib.request
from pathlib import Path

log_path = Path("/content/lax.log")
log_handle = log_path.open("w")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true",
    ],
    cwd=str(REPO_DIR),
    env=os.environ.copy(),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Streamlit PID:", streamlit_process.pid)
print("Waiting for the local server (the first model load can take time)...")

ready = False
for attempt in range(45):
    time.sleep(2)
    if streamlit_process.poll() is not None:
        print("Streamlit exited with code:", streamlit_process.returncode)
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass

print("Local Streamlit health check passed:", ready)
print("\n--- Recent LAX log output ---")
print(log_path.read_text(errors="replace")[-10000:])

if not ready:
    raise RuntimeError("Streamlit did not become ready. Read the log above before retrying.")

## 9. Create a temporary public link using Cloudflare Tunnel

Run this cell last. It keeps the tunnel open while you use LAX. Copy the new `https://….trycloudflare.com` URL from the output.

In [ ]:
import re
import shutil
import subprocess
import time
from pathlib import Path
import urllib.request

# Install cloudflared automatically if the earlier install cell was skipped
# or failed. Uses the Debian package and lets apt resolve dependencies.
cloudflared = shutil.which("cloudflared")
if cloudflared is None:
    deb_path = Path("/content/cloudflared-linux-amd64.deb")
    download_url = (
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64.deb"
    )

    print("cloudflared is missing. Downloading the official package...")
    subprocess.run(
        ["wget", "-q", "--show-progress", download_url, "-O", str(deb_path)],
        check=True,
    )

    if not deb_path.is_file() or deb_path.stat().st_size < 1_000_000:
        raise RuntimeError(
            "The cloudflared package download appears incomplete. "
            "Retry this cell or inspect network access."
        )

    print("Installing cloudflared and resolving package dependencies...")
    subprocess.run(
        ["apt-get", "install", "-y", str(deb_path)],
        check=True,
    )
    cloudflared = shutil.which("cloudflared")

if cloudflared is None:
    raise RuntimeError(
        "cloudflared still was not found after installation. "
        "Run !dpkg -l cloudflared and share the output."
    )

version = subprocess.run(
    [cloudflared, "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print("Installed:", version.stdout.strip())

# Confirm Streamlit is answering locally before opening the public tunnel.
try:
    with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=5) as response:
        if response.status != 200:
            raise RuntimeError(f"Streamlit health check returned HTTP {response.status}")
except Exception as exc:
    raise RuntimeError(
        "Streamlit is not healthy at http://127.0.0.1:8501. "
        "Run the Streamlit startup cell and inspect /content/lax.log before starting the tunnel."
    ) from exc

log_path = Path("/content/cloudflared.log")
log_path.write_text("", encoding="utf-8")
log_handle = log_path.open("a", encoding="utf-8")

cloudflared_process = subprocess.Popen(
    [
        cloudflared, "tunnel",
        "--url", "http://127.0.0.1:8501",
        "--protocol", "http2",
        "--loglevel", "info",
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

print("Cloudflare Tunnel started in the background.")
print("Process ID:", cloudflared_process.pid)
print("Checking for a public URL (up to 120 seconds)...")

public_url = None
deadline = time.time() + 120
while time.time() < deadline:
    if cloudflared_process.poll() is not None:
        break

    logs = log_path.read_text(encoding="utf-8", errors="replace")
    found = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if found:
        public_url = found[-1]
        break
    time.sleep(2)

if public_url:
    print()
    print("LAX public URL:", public_url)
    print("Open the URL in a new browser tab.")
    print("Keep this Colab runtime active while using LAX.")
else:
    logs = log_path.read_text(encoding="utf-8", errors="replace")
    print("No public URL appeared within 120 seconds.")
    print()
    print("--- Last Cloudflare logs ---")
    print(logs[-12000:] if logs else "(Cloudflare log is empty)")
    if cloudflared_process.poll() is not None:
        print("Cloudflare exited with code:", cloudflared_process.returncode)
    print()
    print("If logs show connection refused to 127.0.0.1:8501, restart Streamlit first.")
    print("For DNS or network errors, retry on a working network.")

## Troubleshooting

- **TorchAO compatibility error:** the CPU launcher removes an old incompatible optional TorchAO version before model startup; if the error remains, share the exact traceback.
- **Host error:** inspect `/content/lax.log`; make sure Streamlit passes its local health check before opening the tunnel.
- **Out of memory:** restart the runtime and run from the top. CPU fast mode uses the smaller 0.5B model and does not load the separate general chat model or the 1.5B LoRA adapter.
- **Slow first response:** the first run may need to download the smaller model and load the embedding model. Later responses should be faster than the 1.5B FP32 CPU path, but speed still depends on Colab CPU resources.
- **Want the original fine-tuned model?** Use the regular launcher with a GPU, or set `LAX_CPU_FAST_MODE=0` and make the saved adapter available; expect substantially slower CPU responses.

LAX is a student wellbeing psychoeducation assistant, not a therapist or emergency service.